# Strands Agents SDK: Building Autonomous Agent Harnesses with Amazon Bedrock

*This notebook explores the **Strands Agents SDK**, detailing how modern agent systems separate the foundational reasoning model from the **runtime harness** (tools, context management, memory, and observability) using **Amazon Nova Pro** (`us.amazon.nova-pro-v1:0`) in region `us-east-2`. Directly adapted and expanded from `plan.md`.*

---

## 1. What is an Agent? Model + Harness

> **Core Insight**: Foundation models are stateless. They process one request, produce a completion, and forget. They cannot take actions, query live private data, or coordinate multi-step workflows alone.

An **Agent** solves this by wrapping the model in an active runtime system:

<div align="center">
  <img src="../images/strands_agent_architecture.svg" alt="Strands Agents SDK Architecture" width="100%" />
</div>

$$\text{Agent} = \text{Foundation Model (Reasoning Engine)} + \text{Agent Harness (Runtime System)}$$

### The Agent Loop
1. **Context Input**: Model receives context (system prompt + user query + tool definitions + conversational history).
2. **Reasoning & Decision**: Model evaluates the goal and decides whether to formulate an answer or call a tool.
3. **Tool Execution**: The harness intercepts the tool call, executes the function or API, and collects the result.
4. **Context Feedback**: The tool output is appended to the context window.
5. **Iteration**: The loop repeats autonomously until the model finishes the task and synthesizes the final response.

### The Three Layers of Agent Engineering
- **1. Prompt Engineering**: Crafting instructions, personas, and behavioral boundaries sent to the model.
- **2. Context Engineering**: Curating and dynamically managing what enters the context window, when, and how.
- **3. Harness Engineering**: Building the runtime system that orchestrates tools, manages execution state, handles errors, and ensures observability.

---

## 2. Setup & Environment Configuration

The **Strands Agents SDK** (`strands-agents`) is an open-source library that provides modular primitives for building agent harnesses.

To install Strands and its utility tools, run the following in your environment:
```bash
pip install strands-agents strands-agents-tools boto3
```

In this section, we initialize our Boto3 Bedrock Runtime client and configure our Amazon Nova model parameters.


In [1]:
import os
import json
import time
import inspect
from botocore.config import Config
from botocore.exceptions import ClientError
import boto3

# Target AWS Region
REGION = "us-east-2"

# Boto3 client configuration for Amazon Bedrock
client_config = Config(
    region_name=REGION,
    connect_timeout=60,
    read_timeout=120,
    retries={"max_attempts": 3, "mode": "standard"}
)

# Initialize Bedrock Runtime client
bedrock_runtime = boto3.client("bedrock-runtime", config=client_config)

# Amazon First-Party Model IDs (Cross-Region System Inference Profiles)
# Configured to Amazon Nova Pro for maximum reasoning accuracy and complex multi-step orchestration
NOVA_PRO  = "us.amazon.nova-pro-v1:0"
NOVA_LITE = "us.amazon.nova-lite-v1:0"
ACTIVE_MODEL = NOVA_PRO

print(f"Target Region:       {REGION}")
print(f"Active Model:        {ACTIVE_MODEL} (Amazon Nova Pro)")
print("Bedrock Runtime Client initialized successfully.")

# Check for Strands Agents SDK installation
try:
    import strands
    from strands import Agent, tool
    from strands.models import BedrockModel
    STRANDS_AVAILABLE = True
    print(f"✅ Strands Agents SDK version {getattr(strands, '__version__', 'detected')} is available.")
except ImportError:
    STRANDS_AVAILABLE = False
    print("ℹ️ Note: 'strands-agents' is not installed in the local environment.")
    print("   The notebook will use a native high-fidelity Boto3 implementation of the Strands Harness pattern.")

Target Region:       us-east-2
Active Model:        us.amazon.nova-pro-v1:0 (Amazon Nova Pro)
Bedrock Runtime Client initialized successfully.
ℹ️ Note: 'strands-agents' is not installed in the local environment.
   The notebook will use a native high-fidelity Boto3 implementation of the Strands Harness pattern.


## 3. Core Concept 1: A Simple Agent (The Autonomous Reasoning Loop)

*Reference: `plan.md` lines 39–51*

Even without external tools, an agent is fundamentally different from a simple text completion API:
- The **harness** establishes the conversational session.
- It formats system messages, enforces token budgets, and listens for model stop sequences.
- It prepares the model to trigger autonomous loops as soon as capabilities are added.

### Code Example: Zero-Tool Agent Loop
```python
from strands import Agent
from strands.models import BedrockModel

agent = Agent(model=BedrockModel(model_id="us.amazon.nova-pro-v1:0"))
response = agent("What are the key differences between REST and GraphQL APIs?")
print(response)
```

In [2]:
# Implementation of Simple Agent Loop
class SimpleStrandsHarness:
    """
    Educational implementation of the Strands Agent Harness runtime
    powered by Amazon Bedrock Nova Pro.
    """
    def __init__(self, model_id=NOVA_PRO, system_prompt=None):
        self.model_id = model_id
        self.system_prompt = system_prompt or "You are a concise, helpful technical AI assistant."
        self.messages = []
        
    def __call__(self, prompt: str) -> str:
        # 1. Append user prompt to context
        self.messages.append({"role": "user", "content": [{"text": prompt}]})
        
        # 2. Invoke the model via Bedrock Converse API
        system_prompts = [{"text": self.system_prompt}]
        response = bedrock_runtime.converse(
            modelId=self.model_id,
            messages=self.messages,
            system=system_prompts
        )
        
        # 3. Extract completion and update context
        output_msg = response["output"]["message"]
        self.messages.append(output_msg)
        answer = "".join(c["text"] for c in output_msg["content"] if "text" in c)
        return answer

print("=== Running Simple Agent with Amazon Nova Pro ===")
simple_agent = SimpleStrandsHarness(model_id=NOVA_PRO)
response = simple_agent("What are the 3 key architectural differences between REST and GraphQL APIs? Keep it to 3 bullet points.")
print(response)

=== Running Simple Agent with Amazon Nova Pro ===
- **Endpoint Structure**: REST uses multiple endpoints for different resources, while GraphQL uses a single endpoint for all queries.
- **Data Fetching**: REST returns fixed, predefined data structures, whereas GraphQL allows clients to request specific fields, reducing over-fetching and under-fetching.
- **Versioning**: REST often requires versioning of endpoints for changes, while GraphQL can evolve without breaking changes by adding new fields or types.


## 4. Core Concept 2: Agent with Tools (`@tool` Decorator)

To give an agent the power to act, the **Strands Agents SDK** provides the `@tool` decorator.

### Key Architectural Concepts:
1. **`@tool` Decorator**: Converts any standard Python function into an agent-callable tool.
2. **Docstrings as Semantic Instructions**: The model reads the function docstring (description, parameter explanations, and return types) to decide *when* and *how* to call the tool.
3. **Type Hints Auto-Generate Input Schemas**: Python type hints (`str`, `int`, `float`, `bool`, `dict`, `list`) are automatically transformed into JSON Schema definitions for the model.
4. **System Prompt Alignment**: Guides the agent on tool usage policies, safety boundaries, and workflow order.

Let's build a realistic **Product Inventory & Pricing Tool** inspired by `plan.md`.


In [3]:
# Sample In-Memory Product Inventory Database
PRODUCT_CATALOG = {
    "LP-101": {"name": "TitanBook Pro 16", "category": "Laptop", "price": 1499.00, "stock": 14, "specs": "Apple M3 / 32GB RAM / 1TB SSD"},
    "LP-102": {"name": "NovaAir Ultra 14", "category": "Laptop", "price": 999.00, "stock": 8, "specs": "Intel Core Ultra 7 / 16GB RAM / 512GB SSD"},
    "MN-201": {"name": "ViewSonic 4K Studio 27", "category": "Monitor", "price": 450.00, "stock": 22, "specs": "4K UHD IPS 144Hz"},
    "MN-202": {"name": "UltraWide Curved 34", "category": "Monitor", "price": 680.00, "stock": 5, "specs": "WQHD 3440x1440 165Hz"},
    "AC-301": {"name": "ErgoPro Wireless Keyboard", "category": "Accessory", "price": 120.00, "stock": 45, "specs": "Split mechanical layout"},
}

DISCOUNT_CODES = {
    "SAVE10": 0.10,
    "TECH15": 0.15,
    "VIP20":  0.20
}

def query_product_database(query: str = "") -> str:
    """Query the internal product catalog for inventory, pricing, and hardware specifications.

    Args:
        query: Search keywords such as category (e.g. 'Laptop', 'Monitor'), model names ('4K', 'TitanBook'), or product IDs ('LP-101'). Leave blank or pass 'all' to list entire catalog.

    Returns:
        A JSON string containing matched products, stock availability, and retail pricing.
    """
    query_lower = (query or "").lower().strip()
    if not query_lower or query_lower in ["all", "*", "everything"]:
        matches = [{"product_id": pid, **pdata} for pid, pdata in PRODUCT_CATALOG.items()]
        return json.dumps({"status": "success", "count": len(matches), "products": matches}, indent=2)

    stop_words = {"the", "a", "an", "in", "our", "under", "catalog", "for", "with", "do", "we", "have", "what", "stock"}
    cleaned = "".join(c if c.isalnum() or c.isspace() else " " for c in query_lower)
    words = [w for w in cleaned.split() if w not in stop_words and len(w) > 1]
    stemmed_words = [w[:-1] if w.endswith("s") and len(w) > 3 and not w.endswith("ss") else w for w in words]

    scored_matches = []
    for pid, pdata in PRODUCT_CATALOG.items():
        text = f"{pid} {pdata['name']} {pdata['category']} {pdata['specs']}".lower()
        score = sum(1 for w in (words + stemmed_words) if w in text)
        if score > 0:
            scored_matches.append((score, {"product_id": pid, **pdata}))

    scored_matches.sort(key=lambda x: x[0], reverse=True)
    matches = [item for _, item in scored_matches]

    if not matches:
        return json.dumps({"status": "not_found", "message": f"No products matching '{query}' were found in catalog."})
    return json.dumps({"status": "success", "count": len(matches), "products": matches}, indent=2)


def calculate_discount(product_id: str, promo_code: str) -> str:
    """Calculate the discounted price for a product given a promotional code.

    Args:
        product_id: The unique product identifier (e.g. 'LP-101', 'LP-102', 'MN-201').
        promo_code: The promotional coupon code (e.g. 'SAVE10', 'TECH15', 'VIP20').

    Returns:
        A JSON string containing original price, discount percentage, and final discounted price.
    """
    product = PRODUCT_CATALOG.get(product_id.upper())
    if not product:
        return json.dumps({"error": f"Product ID '{product_id}' not found."})
        
    code_upper = promo_code.upper().strip()
    discount_rate = DISCOUNT_CODES.get(code_upper)
    if discount_rate is None:
        return json.dumps({"error": f"Invalid or expired promo code: '{promo_code}'."})
        
    original_price = product["price"]
    discount_amount = round(original_price * discount_rate, 2)
    final_price = round(original_price - discount_amount, 2)
    
    return json.dumps({
        "product_id": product_id.upper(),
        "product_name": product["name"],
        "original_price": original_price,
        "promo_code": code_upper,
        "discount_percent": f"{int(discount_rate * 100)}%",
        "discount_amount": discount_amount,
        "final_price": final_price
    }, indent=2)

print("Tool functions defined with type hints and docstrings:")
print(f"• query_product_database: {query_product_database.__doc__.strip().splitlines()[0]}")
print(f"• calculate_discount:     {calculate_discount.__doc__.strip().splitlines()[0]}")

Tool functions defined with type hints and docstrings:
• query_product_database: Query the internal product catalog for inventory, pricing, and hardware specifications.
• calculate_discount:     Calculate the discounted price for a product given a promotional code.


### Automatic Schema Generation from Python Types

In the Strands framework, the harness automatically inspects Python function signatures and type annotations to generate standard JSON schemas for the foundation model.

Below, we convert Python type annotations into Bedrock-compliant tool specifications.


In [4]:
def function_to_bedrock_tool_spec(func):
    """
    Inspects a Python function's docstrings and type annotations to generate
    an Amazon Bedrock Converse API tool specification (mirroring the Strands @tool decorator).
    """
    sig = inspect.signature(func)
    doc = inspect.getdoc(func) or ""
    
    # Extract brief description from docstring (first paragraph)
    description = doc.split("\n\n")[0].replace("\n", " ").strip()
    
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        dict: "object",
        list: "array"
    }
    
    properties = {}
    required = []
    
    for param_name, param in sig.parameters.items():
        param_type = type_map.get(param.annotation, "string")
        properties[param_name] = {
            "type": param_type,
            "description": f"Parameter '{param_name}' for {func.__name__}"
        }
        if param.default == inspect.Parameter.empty:
            required.append(param_name)
            
    return {
        "toolSpec": {
            "name": func.__name__,
            "description": description,
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": properties,
                    "required": required
                }
            }
        }
    }

# Build tool specifications for our agent
registered_tools = [query_product_database, calculate_discount]
tools_dict = {f.__name__: f for f in registered_tools}
bedrock_tool_specs = [function_to_bedrock_tool_spec(f) for f in registered_tools]

print("Auto-generated Bedrock Tool Specifications:")
print(json.dumps(bedrock_tool_specs, indent=2))


Auto-generated Bedrock Tool Specifications:
[
  {
    "toolSpec": {
      "name": "query_product_database",
      "description": "Query the internal product catalog for inventory, pricing, and hardware specifications.",
      "inputSchema": {
        "json": {
          "type": "object",
          "properties": {
            "query": {
              "type": "string",
              "description": "Parameter 'query' for query_product_database"
            }
          },
          "required": []
        }
      }
    }
  },
  {
    "toolSpec": {
      "name": "calculate_discount",
      "description": "Calculate the discounted price for a product given a promotional code.",
      "inputSchema": {
        "json": {
          "type": "object",
          "properties": {
            "product_id": {
              "type": "string",
              "description": "Parameter 'product_id' for calculate_discount"
            },
            "promo_code": {
              "type": "string",
             

## 5. Executing the ReAct Loop with Amazon Nova Pro

Now we implement the autonomous **ReAct** (Reasoning + Acting) execution loop using **Amazon Nova Pro** (`us.amazon.nova-pro-v1:0`):
1. Model reviews user intent and queries the product catalog.
2. The harness intercepts the `tool_use` request and calls `query_product_database`.
3. The catalog result is returned into the context window as a `toolResult`.
4. The model identifies the matching laptop and applies `calculate_discount`.
5. The final answer is synthesized and presented to the user.

In [5]:
class StrandsAgentHarness:
    """
    Production-ready Agent Harness implementing the Strands architecture:
    - Dynamic tool execution
    - Conversational context retention
    - Multi-step ReAct loops with Amazon Nova Pro
    - Observability and trace streaming
    """
    def __init__(self, model_id=NOVA_PRO, tools=None, system_prompt=None, max_iterations=6):
        self.model_id = model_id
        self.tools = {f.__name__: f for f in (tools or [])}
        self.tool_specs = [function_to_bedrock_tool_spec(f) for f in (tools or [])]
        self.system_prompt = system_prompt or "You are an intelligent product specialist and pricing assistant."
        self.max_iterations = max_iterations
        self.conversation_history = []
        
    def run(self, user_prompt: str, verbose: bool = True) -> str:
        if verbose:
            print(f"\n{'='*70}")
            print(f"👤 USER PROMPT: '{user_prompt}'")
            print(f"{'='*70}")
            
        self.conversation_history.append({"role": "user", "content": [{"text": user_prompt}]})
        system_prompts = [{"text": self.system_prompt}]
        
        iteration = 0
        while iteration < self.max_iterations:
            iteration += 1
            if verbose:
                print(f"\n[Harness Iteration {iteration}]: Invoking {self.model_id}...")
                
            response = bedrock_runtime.converse(
                modelId=self.model_id,
                messages=self.conversation_history,
                system=system_prompts,
                toolConfig={"tools": self.tool_specs} if self.tool_specs else None
            )
            
            output_msg = response["output"]["message"]
            self.conversation_history.append(output_msg)
            stop_reason = response["stopReason"]
            
            if stop_reason == "tool_use":
                # Find all tool requests
                tool_requests = [c["toolUse"] for c in output_msg["content"] if "toolUse" in c]
                tool_results = []
                
                for tool_req in tool_requests:
                    func_name = tool_req["name"]
                    tool_input = tool_req["input"]
                    tool_use_id = tool_req["toolUseId"]
                    
                    if verbose:
                        print(f"  🤖 MODEL DECISION: Call tool '{func_name}'")
                        print(f"  📥 ARGUMENTS:     {tool_input}")
                        
                    if func_name in self.tools:
                        try:
                            result_str = self.tools[func_name](**tool_input)
                            tool_result_content = {"text": result_str}
                            if verbose:
                                print(f"  ⚡ TOOL EXECUTED:  {result_str[:120]}...")
                        except Exception as e:
                            tool_result_content = {"text": json.dumps({"error": str(e)})}
                    else:
                        tool_result_content = {"text": json.dumps({"error": f"Tool '{func_name}' not found."})}
                        
                    tool_results.append({
                        "toolResult": {
                            "toolUseId": tool_use_id,
                            "content": [tool_result_content]
                        }
                    })
                    
                # Feed tool results back into the context window
                self.conversation_history.append({"role": "user", "content": tool_results})
            else:
                # Model produced final answer
                final_answer = "".join(c["text"] for c in output_msg["content"] if "text" in c)
                if verbose:
                    print(f"\n✨ FINAL SYNTHESIS:\n{final_answer.strip()}\n")
                return final_answer
                
        return "Max agent loop iterations reached without completion."

# Initialize and test the agent with Amazon Nova Pro
hardware_agent = StrandsAgentHarness(
    model_id=NOVA_PRO,
    tools=[query_product_database, calculate_discount],
    system_prompt="You are a professional IT hardware procurement advisor. Recommend products with pricing and apply discount codes when requested."
)

query = "What laptops do we have in stock under $1200, and what would the final price be if I apply promo code TECH15?"
final_response = hardware_agent.run(query)


👤 USER PROMPT: 'What laptops do we have in stock under $1200, and what would the final price be if I apply promo code TECH15?'

[Harness Iteration 1]: Invoking us.amazon.nova-pro-v1:0...
  🤖 MODEL DECISION: Call tool 'query_product_database'
  📥 ARGUMENTS:     {'query': 'laptops under $1200'}
  ⚡ TOOL EXECUTED:  {
  "status": "success",
  "count": 2,
  "products": [
    {
      "product_id": "LP-101",
      "name": "TitanBook Pro ...

[Harness Iteration 2]: Invoking us.amazon.nova-pro-v1:0...
  🤖 MODEL DECISION: Call tool 'calculate_discount'
  📥 ARGUMENTS:     {'product_id': 'LP-102', 'promo_code': 'TECH15'}
  ⚡ TOOL EXECUTED:  {
  "product_id": "LP-102",
  "product_name": "NovaAir Ultra 14",
  "original_price": 999.0,
  "promo_code": "TECH15",
 ...

[Harness Iteration 3]: Invoking us.amazon.nova-pro-v1:0...

✨ FINAL SYNTHESIS:
The NovaAir Ultra 14 laptop is currently in stock and is priced under $1200. After applying the promo code TECH15, the final price is $849.15.



## 6. Out-of-the-Box Agents & Preconfigured Toolkits

Strands also ships with preconfigured defaults that provide a capable agent out-of-the-box:
- **Built-in tools**: File operations (`file_read`, `file_write`), code editing (`editor`), bash shell execution (`shell`), and web access (`http_request`).
- **Sub-Agent Delegation (`use_agent`)**: Spawns specialized child agents for isolated subtasks.
- **Model Agnostic**: Seamlessly binds to Amazon Bedrock models (`BedrockModel`).

Let's implement file operation tools and a delegated sub-agent to demonstrate this capability.


In [6]:
# Implementing Out-of-the-Box Tool Primitives (file_read, file_write, use_agent)

def file_write(filename: str, content: str) -> str:
    """Write text content to a local file.

    Args:
        filename: Name or path of the file to create or overwrite.
        content: Text content to write into the file.

    Returns:
        Confirmation message with file size in bytes.
    """
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    return json.dumps({"status": "success", "file": filename, "bytes_written": len(content)})


def file_read(filename: str) -> str:
    """Read and return text content from a local file.

    Args:
        filename: Name or path of the file to read.
        Returns:
        The content of the file or an error message if not found.
    """
    if not os.path.exists(filename):
        return json.dumps({"status": "error", "message": f"File '{filename}' does not exist."})
    with open(filename, "r", encoding="utf-8") as f:
        data = f.read()
    return json.dumps({"status": "success", "file": filename, "content": data})


def use_agent(subagent_role: str, task: str) -> str:
    """Delegate a specialized subtask to a focused child sub-agent.

    Args:
        subagent_role: The role/persona of the child agent (e.g. 'TechnicalWriter', 'FinancialAuditor').
        task: The specific task description for the sub-agent.

    Returns:
        The sub-agent's completed analysis.
    """
    print(f"\n  🔀 [Sub-Agent Delegation]: Spawning '{subagent_role}' on Amazon Nova Pro...")
    child_agent = StrandsAgentHarness(
        model_id=NOVA_PRO,
        system_prompt=f"You are a specialized sub-agent acting as a {subagent_role}. Perform the requested task directly and concisely without requesting additional tools."
    )
    result = child_agent.run(task, verbose=False)
    return json.dumps({"subagent": subagent_role, "result": result.strip()})

print("Built-in utility tools configured on Amazon Nova Pro:")
print("• file_write: Local file persistence")
print("• file_read:  File retrieval")
print("• use_agent:  Autonomous sub-agent delegation")

Built-in utility tools configured on Amazon Nova Pro:
• file_write: Local file persistence
• file_read:  File retrieval
• use_agent:  Autonomous sub-agent delegation


### Testing File Creation and Sub-Agent Delegation

In this task, the primary agent receives a request to:
1. Lookup a monitor in the database.
2. Delegate the drafting of an executive purchasing justification to a `TechnicalWriter` sub-agent.
3. Save the executive report to a file named `purchase_order_summary.txt`.


In [7]:
# Initialize agent equipped with both database tools and system tools on Amazon Nova Pro
executive_agent = StrandsAgentHarness(
    model_id=NOVA_PRO,
    tools=[query_product_database, file_write, file_read, use_agent],
    system_prompt="""You are an autonomous operations director.
Follow this plan:
1. Search the catalog for the requested item using query_product_database.
2. Delegate the justification drafting to the requested sub-agent using use_agent with the product details.
3. Save the resulting justification to the specified file using file_write.
4. Output your final confirmation message once the file is written."""
)

procurement_prompt = (
    "Find the 4K monitor in our catalog. Then delegate to a 'TechnicalWriter' sub-agent "
    "to write a 2-sentence purchasing justification. Finally, save that justification into 'purchase_order_summary.txt'."
)

result = executive_agent.run(procurement_prompt)

# Verify the file was written to disk
if os.path.exists("purchase_order_summary.txt"):
    with open("purchase_order_summary.txt", "r") as f:
        print("📄 Content of 'purchase_order_summary.txt':")
        print(f.read())


👤 USER PROMPT: 'Find the 4K monitor in our catalog. Then delegate to a 'TechnicalWriter' sub-agent to write a 2-sentence purchasing justification. Finally, save that justification into 'purchase_order_summary.txt'.'

[Harness Iteration 1]: Invoking us.amazon.nova-pro-v1:0...
  🤖 MODEL DECISION: Call tool 'query_product_database'
  📥 ARGUMENTS:     {'query': '4K monitor'}
  ⚡ TOOL EXECUTED:  {
  "status": "success",
  "count": 2,
  "products": [
    {
      "product_id": "MN-201",
      "name": "ViewSonic 4K S...

[Harness Iteration 2]: Invoking us.amazon.nova-pro-v1:0...
  🤖 MODEL DECISION: Call tool 'use_agent'
  📥 ARGUMENTS:     {'task': 'Write a 2-sentence purchasing justification for the ViewSonic 4K Studio 27 monitor with 4K UHD IPS 144Hz specs.', 'subagent_role': 'TechnicalWriter'}

  🔀 [Sub-Agent Delegation]: Spawning 'TechnicalWriter' on Amazon Nova Pro...

[Harness Iteration 3]: Invoking us.amazon.nova-pro-v1:0...
  🤖 MODEL DECISION: Call tool 'use_agent'
  📥 ARGUMENTS:     {

## 7. Core Concept 4: Context Engineering & Automatic Context Management

As agents run multi-step loops and handle large tool outputs, context windows face three critical challenges:
1. **Context Window Exhaustion**: Running out of max token capacity.
2. **Attention Degradation (Needle in a Haystack)**: Reasoning accuracy drops when bloated histories dilute key facts.
3. **Escalating Costs & Latency**: Processing tens of thousands of tokens per turn slows TTFT (Time to First Token).

### Strands Context Management Primitives:
- **`SummarizingConversationManager`**: Compresses older conversational turns into a high-density executive summary while keeping recent turns intact.
- **`ContextOffloader` & `FileStorage`**: Intercepts massive tool outputs (such as a 100KB API payload or SQL dump) *before* they enter the context window, offloads the raw data to local disk or S3, and injects a compact preview with references.


In [8]:
class ContextOffloader:
    """
    Plugin that intercepts large tool responses, persists them to disk,
    and returns a compact preview to avoid context window bloat.
    """
    def __init__(self, storage_dir="context_storage", max_chars=300):
        self.storage_dir = storage_dir
        self.max_chars = max_chars
        os.makedirs(storage_dir, exist_ok=True)
        
    def offload_if_large(self, tool_name: str, raw_output: str) -> str:
        if len(raw_output) <= self.max_chars:
            return raw_output
            
        file_id = f"offload_{tool_name}_{int(time.time()*1000)}.txt"
        file_path = os.path.join(self.storage_dir, file_id)
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(raw_output)
            
        preview = raw_output[:self.max_chars] + "... [TRUNCATED]"
        return json.dumps({
            "_offloaded": True,
            "original_length": len(raw_output),
            "stored_at": file_path,
            "preview": preview,
            "note": "Full content offloaded to avoid context bloat. Read file if full data is required."
        }, indent=2)


class SummarizingConversationManager:
    """
    Proactive conversation manager that compresses older conversational turns
    when the message history exceeds a threshold using Amazon Nova Pro.
    """
    def __init__(self, max_turns_before_summary=4, model_id=NOVA_PRO):
        self.max_turns_before_summary = max_turns_before_summary
        self.model_id = model_id
        
    def compact_history(self, messages: list) -> list:
        if len(messages) <= self.max_turns_before_summary:
            return messages
            
        print(f"\n  ✂️ [Context Engineering]: Message count ({len(messages)}) exceeded threshold ({self.max_turns_before_summary}).")
        print(f"     Summarizing older history to preserve token budget using {self.model_id}...")
        
        # Keep the latest 2 turns intact, summarize older turns
        older_messages = messages[:-2]
        recent_messages = messages[-2:]
        
        # Build prompt to summarize
        summary_prompt = "Summarize the key facts, decisions, and data from this conversation history in 3 concise bullet points:\n"
        for m in older_messages:
            role = m["role"]
            text_blocks = [c.get("text", "") for c in m.get("content", []) if "text" in c]
            summary_prompt += f"{role.upper()}: {' '.join(text_blocks)}\n"
            
        summary_resp = bedrock_runtime.converse(
            modelId=self.model_id,
            messages=[{"role": "user", "content": [{"text": summary_prompt}]}]
        )
        summary_text = summary_resp["output"]["message"]["content"][0]["text"].strip()
        print(f"     ✅ Summary Generated: {summary_text[:120]}...\n")
        
        compacted = [
            {"role": "user", "content": [{"text": f"[CONVERSATION SUMMARY OF PRIOR TURNS]:\n{summary_text}"}]},
            {"role": "assistant", "content": [{"text": "Understood. I retain this summary of prior context."}]}
        ] + recent_messages
        
        return compacted

print("Context Management primitives ready with Amazon Nova Pro:")
print("• ContextOffloader: Large payload interception")
print("• SummarizingConversationManager: Dynamic history compaction")

Context Management primitives ready with Amazon Nova Pro:
• ContextOffloader: Large payload interception
• SummarizingConversationManager: Dynamic history compaction


### Demonstrating Context Management in Action

Let's test our `SummarizingConversationManager` across multiple consecutive conversational turns. Watch how the harness proactively summarizes older turns while preserving recent state.


In [9]:
# Create managed agent with proactive context summarization on Amazon Nova Pro
managed_agent = StrandsAgentHarness(
    model_id=NOVA_PRO,
    tools=[query_product_database, calculate_discount],
    system_prompt="You are a helpful IT assistant. Remember user choices across turns."
)
conv_manager = SummarizingConversationManager(max_turns_before_summary=4, model_id=NOVA_PRO)

# Turn 1
print("=== TURN 1 ===")
managed_agent.run("Hi! I'm Alice from the Engineering department.", verbose=False)

# Turn 2
print("=== TURN 2 ===")
managed_agent.run("What laptops do we have in stock?", verbose=False)

# Turn 3
print("=== TURN 3 ===")
managed_agent.run("I am interested in the TitanBook Pro 16.", verbose=False)

print(f"\nCurrent conversation history length before compaction: {len(managed_agent.conversation_history)} messages.")

# Apply context compaction
managed_agent.conversation_history = conv_manager.compact_history(managed_agent.conversation_history)

print(f"Compacted conversation history length: {len(managed_agent.conversation_history)} messages.")

# Turn 4: Verify recall of Alice's identity and choice after compaction
print("\n=== TURN 4: Testing Recall After Compaction ===")
recall_response = managed_agent.run("Can you remind me what my name is and which laptop I was looking at?", verbose=False)
print(f"Agent Response:\n{recall_response.strip()}")

=== TURN 1 ===
=== TURN 2 ===
=== TURN 3 ===

Current conversation history length before compaction: 8 messages.

  ✂️ [Context Engineering]: Message count (8) exceeded threshold (4).
     Summarizing older history to preserve token budget using us.amazon.nova-pro-v1:0...
     ✅ Summary Generated: - Alice from the Engineering department inquired about the laptops currently in stock.
- The available laptops are the T...

Compacted conversation history length: 4 messages.

=== TURN 4: Testing Recall After Compaction ===
Agent Response:
<thinking> The user did not provide a name previously, but they mentioned interest in the TitanBook Pro 16. I need to respond with the information they asked for. </thinking>

Your name is Alice from the Engineering department, and you were looking at the TitanBook Pro 16. This laptop has the following details:

- **Product ID:** LP-101
- **Price:** $1499.00
- **Stock:** 14 units
- **Specs:** Apple M3 / 32GB RAM / 1TB SSD

Is there anything else you would 

## 8. Clean Up Resources

Because the Strands Agents SDK runs client-side (or on standard serverless containers like AWS Lambda or AgentCore Runtime), it does not create billable persistent control-plane resources on AWS.

We clean up the local sample files and temporary context directories created during the exercise.


In [10]:
# Cleanup local files created during this notebook
files_to_remove = ["purchase_order_summary.txt"]
directories_to_clean = ["context_storage"]

print("Starting cleanup...")
for filename in files_to_remove:
    if os.path.exists(filename):
        os.remove(filename)
        print(f"  • Removed file: {filename}")

for directory in directories_to_clean:
    if os.path.exists(directory):
        for f in os.listdir(directory):
            os.remove(os.path.join(directory, f))
        os.rmdir(directory)
        print(f"  • Cleaned directory: {directory}")

print("Cleanup completed successfully!")


Starting cleanup...
  • Removed file: purchase_order_summary.txt
Cleanup completed successfully!


## Summary & Key Takeaways

In this notebook, you explored the **Strands Agents SDK** and learned the foundational architecture of modern AI agent systems:

### Key Architectural Concepts Mastered:
1. **Model + Harness Architecture**: Models are stateless reasoning engines; the harness orchestrates the reasoning loop, tool execution, and context.
2. **Three Engineering Layers**: Prompt Engineering (instructions), Context Engineering (what enters the token window), and Harness Engineering (the runtime loop).
3. **The `@tool` Decorator**: Turned standard Python functions into agent tools with auto-generated JSON schemas and docstring-derived descriptions.
4. **Out-of-the-Box Toolkits**: Integrated file operations (`file_write`, `file_read`) and sub-agent delegation (`use_agent`).
5. **Context Management**: Implemented `SummarizingConversationManager` to compress chat histories and `ContextOffloader` to intercept oversized tool payloads.
6. **Amazon Bedrock & Nova Models**: Powered all agent reasoning loops using **Amazon Nova Pro** (`us.amazon.nova-pro-v1:0`) in region `us-east-2`.

---

### Complete Amazon Bedrock Learning Series
- [001-Invoke-model.ipynb](001-Invoke-model.ipynb): Synchronous invocations, prompt engineering, and Titan Image Generation.
- [001b-Converse-API.ipynb](001b-Converse-API.ipynb): Unified Converse API, multi-turn chat, and multimodal image/document analysis.
- [002-Streaming-response.ipynb](002-Streaming-response.ipynb): Real-time streaming tokens, TTFT performance, and generator patterns.
- [003-Guardrails.ipynb](003-Guardrails.ipynb): Content filtering, PII redaction, topic filtering, and programmatic Guardrail APIs.
- [004-Knowledge-Base.ipynb](004-Knowledge-Base.ipynb): Retrieval-Augmented Generation (RAG) and Titan Text Embeddings V2.
- [005-Prompt-Management.ipynb](005-Prompt-Management.ipynb): Reusable prompt templates, versioning, and prompt optimization.
- [006-Tool-Calling.ipynb](006-Tool-Calling.ipynb): Math calculations and AWS Lambda tool calling.
- [007-Tool-Calling-Converse-API.ipynb](007-Tool-Calling-Converse-API.ipynb): In-depth Converse API building blocks, weather tool calling, and context window management.
- [008-Bedrock-Agents.ipynb](008-Bedrock-Agents.ipynb): Autonomous agent orchestration, action groups, code interpretation, memory, and AgentCore.
- [009-Prompt-Caching.ipynb](009-Prompt-Caching.ipynb): Converse API prompt caching for latency reduction and cost optimization.
- [010-Strands-Agent.ipynb](010-Strands-Agent.ipynb): Strands Agents SDK, Model + Harness design, `@tool` decorators, context engineering, and sub-agent delegation with Amazon Nova Pro.